In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

In [3]:
fecha_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=2", con=connection2)
fecha_ini= fecha_ini.iloc[0, 0]

fecha_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_mod=2", con=connection2)
fecha_fin= fecha_fin.iloc[0, 0]

In [4]:
dias_por_intervalo = 15

# Inicializa la fecha actual
fecha_actual = fecha_ini

while fecha_actual <= fecha_fin:
	inicioTiempo = time.time()
	now_inicio = datetime.now()

	fecha_ini_intervalo = fecha_actual
	fecha_fin_intervalo = fecha_actual + timedelta(days=dias_por_intervalo - 1)

	if fecha_fin_intervalo > fecha_fin:
		fecha_fin_intervalo = fecha_fin

	fecha_ini_str = fecha_ini_intervalo.strftime('%Y-%m-%d')
	fecha_fin_str = (fecha_fin_intervalo + timedelta(days=1)).strftime('%Y-%m-%d')
	fecha_fin_display_str = fecha_fin_intervalo.strftime('%Y-%m-%d')

	print(f"Procesando de {fecha_ini_str} al {fecha_fin_display_str}")

	now1 = datetime.now()
	now2 = datetime.now().strftime('%Y-%m-%d')

	query=f"UPDATE etl_act SET fec_act ='{now1}' WHERE id_mod=2"

	c1= text(query)
	connection2.execute(c1)

	tabla='ctaam10'
	col_tabla = "atenambatenfec"
	dat= "cext02_essi"
	col_dat='fec_ate'


	oracledb.init_oracle_client()
	oracledb.version = "8.3.0"
	sys.modules["cx_Oracle"] = oracledb
	un = config("USER4_BDI_POSTGRES")
	pw = config("PASS4_BDI_POSTGRES")
	hostname=config("HOST4_BDI_POSTGRES")
	service_name="WNET"
	port = 1527

	engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
			"host": hostname,
			"port": port,
			"service_name": service_name
		}
	)

	connection0 = engine0.connect()


	query1 = f"""
	SELECT
		c.ATENAMBORICENASICOD,
		c.ATENAMBCENASICOD,
		c.ATENAMBNUM,
		c.ATENAMBATENFEC,
		c.ATENAMBTIPCONCOD,
		c.ATENAMBCSECOD,
		c.CPSCOD,
		c.ATENAMBNUMATECOD,
		c.TIPCONTLEYCOD,
		c.RESATENAMBUCOD,
		c.ATENAMBESTREGCOD,
		c.ATENAMBCREFEC,
		c.ATENAMBMODFEC,
		c.ATENAMBULTREGFEC,
		c.ATENAMBPACSECNUM,
		d.DIAGCOD,
		d.CONDDIAGCOD,
		d.ATENAMBDIAGORD,
		d.ATENAMBTIPODIAGCOD,
		d.ATENAMBCASODIAGCOD,
		d.DIAGATENAMBALTAFLAG
	FROM {tabla} c
	LEFT OUTER JOIN CTDAA10 d		 
		ON c.ATENAMBORICENASICOD = d.ATENAMBORICENASICOD
		AND c.ATENAMBCENASICOD = d.ATENAMBCENASICOD 
		AND c.ATENAMBNUM = d.ATENAMBNUM
	WHERE c.{col_tabla} >= TO_DATE('{fecha_ini_str}', 'YYYY-MM-DD') AND c.{col_tabla} < TO_DATE('{fecha_fin_str}', 'YYYY-MM-DD')
	"""

	base1 = pd.read_sql_query(query1, con=connection0)
	connection0.close()
	engine0.dispose()

	base1 = base1.rename(columns={
		'atenamboricenasicod': 'cod_ori',
		'atenambcenasicod': 'cod_cas',
		'atenambnum': 'ate_num',
		'atenambatenfec': 'fec_ate',
		'atenambtipconcod': 'cod_tipcon',
		'atenambcsecod': 'cod_cse',
		'cpscod': 'cod_cps',
		'atenambnumatecod': 'ate_cod',
		'tipcontleycod': 'ley_cod',
		'resatenambucod': 'res_cod',
		'atenambestregcod': 'est_reg',
		'atenambcrefec': 'fec_cre',
		'atenambmodfec': 'fec_mod',
		'atenambultregfec': 'fec_ult_reg',
		'atenambpacsecnum': 'pac_sec',
		'diagcod': 'cod_diag',
		'conddiagcod': 'cond_diag',
		'atenambdiagord': 'ord_diag',
		'atenambtipodiagcod': 'tipo_diag',
		'atenambcasodiagcod': 'caso_diag',
		'diagatenambaltaflag': 'alta_flg'
	})

	base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

	oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
	oricas['ori_cod'] = oricas['ori_cod'].str.strip()
	base1['cod_ori'] = base1['cod_ori'].str.strip()
	oricas=oricas.rename(columns={"ori_cod":"cod_ori"})
	base1=pd.merge(base1,oricas,how='left',on='cod_ori')

	base1=base1.drop('cod_ori',axis=1)
	
	cas = pd.read_sql_query(f"SELECT id_cas,cod_cas, cod_red FROM dim_cas where cod_red is not null", con=connection2)
	cas = cas.dropna()
	cas['cod_cas'] = cas['cod_cas'].str.strip()
	base1['cod_cas'] = base1['cod_cas'].str.strip()
	base1=pd.merge(base1,cas,how='left',on='cod_cas')

	base1=base1.drop('cod_cas',axis=1)
	
	red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)
	red['cod_red'] = red['cod_red'].str.strip()
	base1['cod_red'] = base1['cod_red'].str.strip()
	base1=pd.merge(base1,red,how='left',on='cod_red')

	base1=base1.drop('cod_red',axis=1)

	tipcon = pd.read_sql_query(f"SELECT id_tipcon,cod_tipcon FROM dim_tipcon", con=connection2)
	tipcon['cod_tipcon'] = tipcon['cod_tipcon'].str.strip()
	base1['cod_tipcon'] = base1['cod_tipcon'].str.strip()
	base1=pd.merge(base1,tipcon,how='left',on='cod_tipcon')

	base1=base1.drop('cod_tipcon',axis=1)

	csep = pd.read_sql_query(f"SELECT id_csep,cod_cse FROM dim_csep", con=connection2)
	csep['cod_cse'] = csep['cod_cse'].str.strip()
	base1['cod_cse'] = base1['cod_cse'].str.strip()
	base1=pd.merge(base1,csep,how='left',on='cod_cse')

	base1=base1.drop('cod_cse',axis=1)

	cps = pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
	cps['cod_cps'] = cps['cod_cps'].str.strip()
	base1['cod_cps'] = base1['cod_cps'].str.strip()
	base1=pd.merge(base1,cps,how='left',on='cod_cps')

	base1=base1.drop('cod_cps',axis=1)

	ley = pd.read_sql_query(f"SELECT id_tipleycont,cod_tipleycont FROM dim_tipleycont", con=connection2)
	ley=ley.rename(columns={"cod_tipleycont":"ley_cod"})
	ley['ley_cod'] = ley['ley_cod'].str.strip()
	base1['ley_cod'] = base1['ley_cod'].str.strip()
	base1=pd.merge(base1,ley,how='left',on='ley_cod')

	base1=base1.drop('ley_cod',axis=1)

	resaten = pd.read_sql_query(f"SELECT id_resaten,cod_resaten FROM dim_resaten", con=connection2)
	resaten=resaten.rename(columns={"cod_resaten":"res_cod"})
	resaten['res_cod'] = resaten['res_cod'].str.strip()
	base1['res_cod'] = base1['res_cod'].str.strip()
	base1=pd.merge(base1,resaten,how='left',on='res_cod')

	base1=base1.drop('res_cod',axis=1)

	estreg = pd.read_sql_query(f"SELECT id_estreg,cod_est FROM dim_estreg", con=connection2)
	estreg=estreg.rename(columns={"cod_est":"est_reg"})
	estreg['est_reg'] = estreg['est_reg'].str.strip()
	base1['est_reg'] = base1['est_reg'].str.strip()
	base1=pd.merge(base1,estreg,how='left',on='est_reg')

	base1=base1.drop('est_reg',axis=1)

	diagcod = pd.read_sql_query(f"SELECT id_cie10,cod_diag FROM dim_cie10", con=connection2)
	diagcod['cod_diag'] = diagcod['cod_diag'].str.strip()
	base1['cod_diag'] = base1['cod_diag'].str.strip()
	base1=pd.merge(base1,diagcod,how='left',on='cod_diag')

	base1=base1.drop('cod_diag',axis=1)

	conddiag = pd.read_sql_query(f"SELECT id_conddiag,cod_conddiag FROM dim_conddiag", con=connection2)
	conddiag=conddiag.rename(columns={"cod_conddiag":"cond_diag"})
	conddiag['cond_diag'] = conddiag['cond_diag'].str.strip()
	base1['cond_diag'] = base1['cond_diag'].str.strip()
	base1=pd.merge(base1,conddiag,how='left',on='cond_diag')

	base1=base1.drop('cond_diag',axis=1)

	tipodiag = pd.read_sql_query(f"SELECT id_tipdiag,cod_tipdiag FROM dim_tipdiag", con=connection2)
	tipodiag=tipodiag.rename(columns={"cod_tipdiag":"tipo_diag"})
	tipodiag['tipo_diag'] = tipodiag['tipo_diag'].str.strip()
	base1['tipo_diag'] = base1['tipo_diag'].str.strip()
	base1=pd.merge(base1,tipodiag,how='left',on='tipo_diag')

	base1=base1.drop('tipo_diag',axis=1)


	casodiag = pd.read_sql_query(f"SELECT id_casodiag,cod_casodiag FROM dim_casodiag", con=connection2)
	casodiag=casodiag.rename(columns={"cod_casodiag":"caso_diag"})
	casodiag['caso_diag'] = casodiag['caso_diag'].str.strip()
	base1['caso_diag'] = base1['caso_diag'].str.strip()
	base1=pd.merge(base1,casodiag,how='left',on='caso_diag')

	base1=base1.drop('caso_diag',axis=1)

	df1_columns = set(base1.columns)
	df2_columns = set(base2.columns) 
	different_columns = df2_columns - df1_columns
	different_columns

	borrando = f"DELETE FROM {dat} WHERE {col_dat} >='{fecha_ini_str}' and {col_dat} <'{fecha_fin_str}'"
	borrado = connection2.execute(borrando)

	comunes = set(base1.columns).intersection(set(base2.columns)) 
	base1 = base1[list(comunes)]
	base1.to_sql(name=f'{dat}', con=engine2, if_exists='append', index=False,chunksize=500000)

	fecha_actual = fecha_fin_intervalo + timedelta(days=1)

	finproceso = time.time()
	tiempoproceso = finproceso - inicioTiempo
	tiempoproceso = round(tiempoproceso, 3)
	print("Proceso completado satisfactoriamente en " + str(tiempoproceso) + " segundos")

query2=f"UPDATE etl_act SET fec_ini ='{now2}' WHERE id_mod=2"
c2= text(query2)
connection2.execute(c2)


Procesando de 2016-01-01 al 2016-01-15
Proceso completado satisfactoriamente en 47.787 segundos
Procesando de 2016-01-16 al 2016-01-30
Proceso completado satisfactoriamente en 54.376 segundos
Procesando de 2016-01-31 al 2016-02-14
Proceso completado satisfactoriamente en 46.652 segundos
Procesando de 2016-02-15 al 2016-02-29
Proceso completado satisfactoriamente en 54.631 segundos
Procesando de 2016-03-01 al 2016-03-15
Proceso completado satisfactoriamente en 54.945 segundos
Procesando de 2016-03-16 al 2016-03-30
Proceso completado satisfactoriamente en 48.039 segundos
Procesando de 2016-03-31 al 2016-04-14
Proceso completado satisfactoriamente en 54.189 segundos
Procesando de 2016-04-15 al 2016-04-29
Proceso completado satisfactoriamente en 55.012 segundos
Procesando de 2016-04-30 al 2016-05-14
Proceso completado satisfactoriamente en 50.678 segundos
Procesando de 2016-05-15 al 2016-05-29
Proceso completado satisfactoriamente en 49.811 segundos
Procesando de 2016-05-30 al 2016-06-13
P

In [5]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()